# บทที่ 8: โครงข่ายประสาทเชิงคอนโวลูชัน (Convolutional Neural Networks: CNN)

ใน Notebook นี้ เราจะสร้างการดำเนินการคอนโวลูชัน (Convolution) การทำพูลลิง (Pooling) และศึกษาโครงสร้างของ CNN

**ศัพท์ที่สำคัญในบทนี้:**

- เวกเตอร์ (vector) — อาร์เรย์หนึ่งมิติ
- เมทริกซ์ (matrix) — อาร์เรย์สองมิติ
- ค่าน้ำหนัก (weight) — พารามิเตอร์ที่ปรับได้
- ค่าไบแอส (bias) — ค่าเลื่อน
- อินพุต (input) — ข้อมูลนำเข้า
- เอาต์พุต (output) — ผลลัพธ์
- ค่าสูญเสีย (loss) — วัดความคลาดเคลื่อน
- เกรเดียนต์ (gradient) — ทิศทางการปรับ
- ฟังก์ชันกระตุ้น (activation function) — ฟังก์ชันไม่เป็นเชิงเส้น
- โครงข่ายประสาทเทียม (neural network) — โมเดลแมชชีนเลิร์นนิง

## 1. นำเข้าไลบรารี

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
try:
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'],
                   capture_output=True)
except FileNotFoundError:
    pass  # เครื่องที่ไม่มี apt-get (macOS/Windows) ใช้ฟอนต์ไทยที่ติดตั้งไว้ในเครื่องแทน

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

np.random.seed(42)

## 2. การดำเนินการคอนโวลูชัน (Convolution Operation)

In [ ]:
def conv2d(image, kernel, stride=1, padding=0):
    """
    การดำเนินการคอนโวลูชัน 2 มิติ

    พารามิเตอร์:
    - image: ภาพอินพุต (H x W)
    - kernel: เคอร์เนล/ฟิลเตอร์ (kH x kW)
    - stride: สไตรด์
    - padding: แพดดิงศูนย์
    """
    # เติมแพดดิง
    if padding > 0:
        image = np.pad(image, padding, mode='constant')

    h, w = image.shape
    kh, kw = kernel.shape

    # ขนาดเอาต์พุต
    out_h = (h - kh) // stride + 1
    out_w = (w - kw) // stride + 1

    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            # ดึงส่วนของภาพ
            region = image[i*stride:i*stride+kh, j*stride:j*stride+kw]
            # คอนโวลูชัน (คูณแต่ละสมาชิกแล้วรวมผล)
            output[i, j] = np.sum(region * kernel)

    return output

# ทดสอบด้วยภาพอย่างง่าย
image = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])

# เคอร์เนลตรวจจับขอบ
kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])

print("=== ภาพต้นฉบับ ===")
print(image)

print("\n=== เคอร์เนล (ตรวจจับขอบแนวตั้ง) ===")
print(kernel)

output = conv2d(image, kernel)
print("\n=== ผลลัพธ์คอนโวลูชัน ===")
print(output)

## 3. การคำนวณขนาดเอาต์พุต (Output Size)

In [ ]:
def calculate_output_size(input_size, kernel_size, stride=1, padding=0):
    """
    คำนวณขนาดเอาต์พุตหลังคอนโวลูชัน

    สูตร: output_size = floor((input_size + 2*padding - kernel_size) / stride) + 1
    """
    output_size = (input_size + 2*padding - kernel_size) // stride + 1
    return output_size

# ตัวอย่าง
print("=== การคำนวณขนาดเอาต์พุต ===")
print(f"อินพุต: 32×32, เคอร์เนล: 3×3, สไตรด์: 1, แพดดิง: 0")
print(f"เอาต์พุต: {calculate_output_size(32, 3, 1, 0)}×{calculate_output_size(32, 3, 1, 0)}")

print(f"\nอินพุต: 32×32, เคอร์เนล: 3×3, สไตรด์: 1, แพดดิง: 1")
print(f"เอาต์พุต: {calculate_output_size(32, 3, 1, 1)}×{calculate_output_size(32, 3, 1, 1)}")

print(f"\nอินพุต: 32×32, เคอร์เนล: 2×2, สไตรด์: 2, แพดดิง: 0")
print(f"เอาต์พุต: {calculate_output_size(32, 2, 2, 0)}×{calculate_output_size(32, 2, 2, 0)}")

## 4. การทำพูลลิง (Pooling)

In [ ]:
def max_pool2d(image, pool_size=2, stride=2):
    """พูลลิงแบบค่าสูงสุด"""
    h, w = image.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1

    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            region = image[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.max(region)

    return output

def avg_pool2d(image, pool_size=2, stride=2):
    """พูลลิงแบบค่าเฉลี่ย"""
    h, w = image.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1

    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            region = image[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.mean(region)

    return output

# ทดสอบ
image = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])

print("=== ภาพต้นฉบับ ===")
print(image)

print("\n=== พูลลิงแบบค่าสูงสุด (2×2) ===")
print(max_pool2d(image))

print("\n=== พูลลิงแบบค่าเฉลี่ย (2×2) ===")
print(avg_pool2d(image))

## 5. ตัวอย่างการตรวจจับขอบ (Edge Detection)

In [ ]:
# สร้างภาพอย่างง่ายที่มีขอบ
image = np.zeros((10, 10))
image[:, 5:] = 1  # ขอบแนวตั้ง

# เคอร์เนลตรวจจับขอบแบบ Sobel
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])  # ตรวจจับขอบแนวตั้ง (ไล่ค่าตามแนวคอลัมน์)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])  # ตรวจจับขอบแนวนอน (ไล่ค่าตามแนวแถว)

edge_x = conv2d(image, sobel_x)
edge_y = conv2d(image, sobel_y)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('ภาพต้นฉบับ')
axes[0].axis('off')

axes[1].imshow(edge_x, cmap='gray')
axes[1].set_title('Sobel X (ขอบแนวตั้ง)')
axes[1].axis('off')

axes[2].imshow(edge_y, cmap='gray')
axes[2].set_title('Sobel Y (ขอบแนวนอน)')
axes[2].axis('off')

axes[3].imshow(np.abs(edge_x) + np.abs(edge_y), cmap='gray')
axes[3].set_title('ขอบรวม')
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 6. การคำนวณจำนวนพารามิเตอร์ของ CNN

In [ ]:
def count_cnn_params(in_channels, out_channels, kernel_size):
    """
    นับจำนวนพารามิเตอร์ในชั้นคอนโวลูชัน

    พารามิเตอร์ = ความสูงเคอร์เนล × ความกว้างเคอร์เนล × ช่องสัญญาณอินพุต × ช่องสัญญาณเอาต์พุต + ค่าไบแอส
    """
    weights = kernel_size * kernel_size * in_channels * out_channels
    biases = out_channels
    return weights + biases

def count_fc_params(in_features, out_features):
    """นับจำนวนพารามิเตอร์ในชั้นเชื่อมต่อเต็มรูปแบบ"""
    return in_features * out_features + out_features

# ตัวอย่าง: สถาปัตยกรรมสไตล์ LeNet-5
print("=== จำนวนพารามิเตอร์ของ CNN ===")
print("\nสถาปัตยกรรมสไตล์ LeNet-5:")

# Conv1: อินพุต 1 ช่องสัญญาณ, เอาต์พุต 6 ช่องสัญญาณ, เคอร์เนล 5×5
params_conv1 = count_cnn_params(1, 6, 5)
print(f"Conv1: {params_conv1} พารามิเตอร์")

# Conv2: อินพุต 6 ช่องสัญญาณ, เอาต์พุต 16 ช่องสัญญาณ, เคอร์เนล 5×5
params_conv2 = count_cnn_params(6, 16, 5)
print(f"Conv2: {params_conv2} พารามิเตอร์")

# FC1: อินพุต 16*5*5, เอาต์พุต 120
params_fc1 = count_fc_params(16*5*5, 120)
print(f"FC1: {params_fc1} พารามิเตอร์")

# FC2: อินพุต 120, เอาต์พุต 84
params_fc2 = count_fc_params(120, 84)
print(f"FC2: {params_fc2} พารามิเตอร์")

# FC3: อินพุต 84, เอาต์พุต 10
params_fc3 = count_fc_params(84, 10)
print(f"FC3: {params_fc3} พารามิเตอร์")

total = params_conv1 + params_conv2 + params_fc1 + params_fc2 + params_fc3
print(f"\nรวมทั้งหมด: {total:,} พารามิเตอร์")

## 7. เปรียบเทียบจำนวนพารามิเตอร์ของ MLP กับ CNN

In [ ]:
# เปรียบเทียบชั้นเชื่อมต่อเต็มรูปแบบ (FC) กับชั้นคอนโวลูชันสำหรับภาพขนาด 224×224×3
# (ตัวอย่างเดียวกับที่แสดงในบท: FC เอาต์พุต 1,000 หน่วย เทียบกับ Conv 3×3 จำนวน 64 ฟิลเตอร์)

input_pixels = 224 * 224 * 3  # คลี่ภาพเป็นเวกเตอร์

# กรณี MLP: ชั้นเชื่อมต่อเต็มรูปแบบจากอินพุตที่คลี่แล้วไปยัง 1,000 หน่วย
fc_units = 1000
params_mlp = count_fc_params(input_pixels, fc_units)
memory_mlp_mib = params_mlp * 4 / (1024 ** 2)

# กรณี CNN: ชั้นคอนโวลูชัน 3×3 รับภาพ 3 ช่องสัญญาณ สร้าง 64 ช่องสัญญาณ
params_cnn = count_cnn_params(in_channels=3, out_channels=64, kernel_size=3)
memory_cnn_kib = params_cnn * 4 / 1024

print("=== เปรียบเทียบพารามิเตอร์ของ MLP กับ CNN (ภาพ 224×224×3) ===")
print(f"อินพุตที่คลี่เป็นเวกเตอร์: {input_pixels:,} ค่า")
print(f"\nMLP (ชั้นเชื่อมต่อเต็มรูปแบบ → {fc_units:,} หน่วย):")
print(f"  พารามิเตอร์: {params_mlp:,}")
print(f"  หน่วยความจำ (float32): {memory_mlp_mib:.0f} MiB")

print(f"\nCNN (ชั้นคอนโวลูชัน 3×3 → 64 ช่องสัญญาณ):")
print(f"  พารามิเตอร์: {params_cnn:,}")
print(f"  หน่วยความจำ (float32): {memory_cnn_kib:.0f} KiB")

print(f"\nอัตราส่วนพารามิเตอร์ MLP ต่อ CNN: ประมาณ {params_mlp / params_cnn:,.0f} : 1")
print("หมายเหตุ: ชั้นคอนโวลูชันใช้ค่าน้ำหนักร่วมกันทั่วภาพ (parameter sharing) และเชื่อมต่อเฉพาะที่ (local connectivity)")
print("จึงมีพารามิเตอร์น้อยกว่ามาก แต่อัตราส่วนนี้ไม่ใช่อัตราส่วนเวลาคำนวณ เพราะคอนโวลูชันต้องประเมินฟิลเตอร์ที่หลายตำแหน่งบนภาพ")

## 8. แบตช์นอร์มัลไลเซชัน (Batch Normalization)

In [ ]:
def batch_norm(x, gamma, beta, eps=1e-5):
    """
    แบตช์นอร์มัลไลเซชันสำหรับหนึ่งช่องสัญญาณ

    x_hat = (x - mu_B) / sqrt(sigma_B^2 + eps)
    y = gamma * x_hat + beta
    """
    mu_B = np.mean(x)
    sigma_B2 = np.var(x)  # ค่าเฉลี่ยของกำลังสองส่วนเบี่ยงเบนในมินิแบตช์ (หารด้วย n)

    x_hat = (x - mu_B) / np.sqrt(sigma_B2 + eps)
    y = gamma * x_hat + beta

    return y, mu_B, sigma_B2, x_hat

# ตัวอย่าง: ค่ากระตุ้นหนึ่งช่องสัญญาณจากมินิแบตช์ขนาด 4
x = np.array([2.0, 4.0, 6.0, 8.0])
gamma, beta = 1.5, 0.5

y, mu_B, sigma_B2, x_hat = batch_norm(x, gamma, beta)

print("=== แบตช์นอร์มัลไลเซชัน ===")
print(f"ค่ากระตุ้นก่อนทำมาตรฐาน: {x}")
print(f"ค่าเฉลี่ยมินิแบตช์ (mu_B): {mu_B}")
print(f"ความแปรปรวนมินิแบตช์ (sigma_B^2): {sigma_B2}")
print(f"ค่าหลังทำมาตรฐาน (x_hat): {x_hat}")
print(f"เอาต์พุตหลังปรับขนาดและเลื่อนด้วย gamma={gamma}, beta={beta}: {y}")
print("หมายเหตุ: ระหว่างการอนุมานจะใช้ค่าเฉลี่ยและความแปรปรวนเคลื่อนที่ที่สะสมระหว่างการฝึกแทนสถิติของมินิแบตช์นี้")

## 9. บล็อกเรซิดวล (Residual Block)

In [ ]:
# ตัวอย่างเดียวกับที่แสดงในบท: การส่งผ่านสัญญาณและเกรเดียนต์ในบล็อกเรซิดวล
# y = F(x; {W_i}) + x  โดย F(x) = W2 @ ReLU(W1 @ x)

def relu(z):
    return np.maximum(0, z)

x = np.array([1.0, 2.0])
W1 = 0.5 * np.eye(2)
W2 = 0.6 * np.eye(2)

h = W1 @ x
F_x = W2 @ relu(h)

y_plain = F_x               # เอาต์พุตของโครงข่ายธรรมดา (ไม่มีทางข้าม)
y_residual = F_x + x        # เอาต์พุตของบล็อกเรซิดวล (มีทางข้าม)

print("=== การส่งผ่านสัญญาณไปข้างหน้า ===")
print(f"h = W1 @ x = {h}")
print(f"F(x) = W2 @ ReLU(h) = {F_x}")
print(f"เอาต์พุตโครงข่ายธรรมดา: y = F(x) = {y_plain}")
print(f"เอาต์พุตบล็อกเรซิดวล: y = F(x) + x = {y_residual}")

# เกรเดียนต์ขาเข้า g = dL/dy
g = np.array([1.0, 1.0])

# ทุกสมาชิกของ h เป็นบวก จึงได้ D_ReLU = I
D_relu = np.eye(2)

grad_plain = W1.T @ D_relu @ W2.T @ g
grad_residual = (W1.T @ D_relu @ W2.T + np.eye(2)) @ g

print("\n=== เกรเดียนต์ต่ออินพุต เมื่อ g = [1, 1] ===")
print(f"โครงข่ายธรรมดา: dL/dx = {grad_plain}")
print(f"บล็อกเรซิดวล: dL/dx = {grad_residual}")
print("หมายเหตุ: พจน์ I จากทางข้ามทำให้เกรเดียนต์มีเส้นทางผ่านโดยตรง ผลลัพธ์นี้เฉพาะตัวอย่างนี้ ไม่ใช่ข้อรับรองขนาดเกรเดียนต์ของทุกบล็อก")

## 10. การเพิ่มปริมาณข้อมูล (Data Augmentation)

In [ ]:
# การแปลงพื้นฐาน: พลิกแนวนอน, หมุน, ปรับความสว่าง (ไม่เปลี่ยนความหมายของป้ายกำกับสำหรับภาพทั่วไป)
image = np.full((10, 10), 0.2)
image[:, 5:] = 0.6  # ภาพระดับเทาสองระดับ (รูปร่างเดียวกับตัวอย่างตรวจจับขอบด้านบน)

flipped = image[:, ::-1]                       # พลิกแนวนอน
rotated = np.rot90(image)                      # หมุน 90 องศา
brightened = np.clip(image * 1.2, 0, 1)        # เพิ่มความสว่างร้อยละ 20

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(
    axes,
    [image, flipped, rotated, brightened],
    ['ภาพต้นฉบับ', 'พลิกแนวนอน', 'หมุน 90 องศา', 'เพิ่มความสว่าง +20%'],
):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

# Mixup: ตัวอย่างและเป้าหมายใหม่ = การผสมเชิงเส้นของสองตัวอย่าง
# x_tilde = lambda*x_i + (1-lambda)*x_j, y_tilde = lambda*y_i + (1-lambda)*y_j
x_i, x_j = np.array([1.0, 0.0]), np.array([0.0, 1.0])   # ภาพตัวอย่าง (ทำให้เรียบง่ายเป็นเวกเตอร์)
y_i, y_j = np.array([1, 0]), np.array([0, 1])           # ป้ายกำกับแบบวันฮอตของสองคลาส
lam = 0.7

x_tilde = lam * x_i + (1 - lam) * x_j
y_tilde = lam * y_i + (1 - lam) * y_j

print("=== Mixup ===")
print(f"lambda = {lam}")
print(f"x_tilde = {lam}×{x_i} + {1-lam:.1f}×{x_j} = {x_tilde}")
print(f"y_tilde = {lam}×{y_i} + {1-lam:.1f}×{y_j} = {y_tilde}")
print("หมายเหตุ: y_tilde ไม่ใช่วันฮอตอีกต่อไป แต่เป็นการแจกแจงเป้าหมายที่ใช้ร่วมกับครอสเอนโทรปีได้โดยตรง")

## 11. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: ผลลัพธ์คอนโวลูชัน (Convolution Output)

In [ ]:
# ให้ image = [[1, 2], [3, 4]] และ kernel = [[1, 0], [0, 1]]
# จงคำนวณผลลัพธ์คอนโวลูชัน

image = np.array([[1, 2], [3, 4]])
kernel = np.array([[1, 0], [0, 1]])

output = np.sum(image * kernel)
print(f"ภาพ:\n{image}")
print(f"\nเคอร์เนล:\n{kernel}")
print(f"\nผลลัพธ์คอนโวลูชัน: {output}")
print(f"การคำนวณ: 1×1 + 2×0 + 3×0 + 4×1 = {output}")

### แบบฝึกหัดที่ 2: ขนาดเอาต์พุต (Output Size)

In [ ]:
# จงคำนวณขนาดเอาต์พุตของ:
# - อินพุต: 224×224
# - เคอร์เนล: 7×7
# - สไตรด์: 2
# - แพดดิง: 3

input_size = 224
kernel_size = 7
stride = 2
padding = 3

output_size = calculate_output_size(input_size, kernel_size, stride, padding)
print(f"ขนาดเอาต์พุต: {output_size}×{output_size}")
print(f"สูตร: floor(({input_size} + 2×{padding} - {kernel_size}) / {stride}) + 1 = {output_size}")

### แบบฝึกหัดที่ 3: พูลลิงแบบค่าสูงสุด (Max Pooling)

In [ ]:
# ให้แผนผังคุณลักษณะ = [[1, 3, 2, 4], [5, 6, 7, 8], [9, 2, 1, 3], [4, 5, 6, 7]]
# จงคำนวณพูลลิงแบบค่าสูงสุดด้วย pool_size=2, stride=2

feature_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 7, 8],
    [9, 2, 1, 3],
    [4, 5, 6, 7]
])

pooled = max_pool2d(feature_map, pool_size=2, stride=2)
print(f"แผนผังคุณลักษณะ:\n{feature_map}")
print(f"\nหลังพูลลิงแบบค่าสูงสุด:\n{pooled}")

### แบบฝึกหัดที่ 4: พารามิเตอร์ของ CNN

In [ ]:
# จงคำนวณพารามิเตอร์ของชั้นคอนโวลูชัน:
# - ช่องสัญญาณอินพุต: 32
# - ช่องสัญญาณเอาต์พุต: 64
# - ขนาดเคอร์เนล: 3×3

in_channels = 32
out_channels = 64
kernel_size = 3

params = count_cnn_params(in_channels, out_channels, kernel_size)
print(f"พารามิเตอร์ของชั้นคอนโวลูชัน:")
print(f"ค่าน้ำหนัก: {kernel_size}×{kernel_size}×{in_channels}×{out_channels} = {kernel_size*kernel_size*in_channels*out_channels}")
print(f"ค่าไบแอส: {out_channels}")
print(f"รวม: {params}")

## บทสรุป

Notebook นี้ครอบคลุม:
1. **การดำเนินการคอนโวลูชัน (Convolution)**: การคำนวณคอนโวลูชันและขนาดเอาต์พุต
2. **พูลลิง (Pooling)**: พูลลิงแบบค่าสูงสุดและค่าเฉลี่ย
3. **การตรวจจับขอบ (Edge Detection)**: การตรวจจับขอบภาพ
4. **การคำนวณพารามิเตอร์**: การคำนวณจำนวนพารามิเตอร์ของ CNN
5. **การเปรียบเทียบ MLP กับ CNN**: แสดงผลของการใช้ค่าน้ำหนักร่วมกันต่อจำนวนพารามิเตอร์
6. **แบตช์นอร์มัลไลเซชัน**: การทำมาตรฐานค่ากระตุ้นด้วยสถิติของมินิแบตช์
7. **บล็อกเรซิดวล**: บทบาทของทางข้ามต่อการส่งผ่านเกรเดียนต์
8. **การเพิ่มปริมาณข้อมูล**: การแปลงภาพพื้นฐานและสูตร Mixup